Meilstone M4

Plan für M4:

1. FastAPI-Backend bauen (Endpunkt /ask), lokal testen — unverändert
2. React-Frontend statt einfachem HTML: einfache Komponente mit Textfeld + Antwortanzeige, die den Backend-Endpunkt aufruft
3. Beide live deployen auf Render (Backend als "Web Service", Frontend als "Static Site")

Backend — Zusammenfassung von 0 bis fertig

1. Ordnerstruktur angelegt

backend/
├── agent.py
├── main.py
└── requirements.txt 2. agent.py geschrieben — die komplette Agent-Logik aus M3, aufgeräumt in wiederverwendbare Bausteine:

- Verbindung zu OpenAI (client) + Chroma (collection) aufgebaut
- search_video() — reine Retrieval-Funktion (Chroma-Suche)
- search_video_tool — derselbe Retriever, mit @tool-Decorator "Agent-lesbar" gemacht
- agent — LangChain-Agent (create_agent) mit gpt-4o-mini + diesem einen Tool
- ask_agent(question) — neue Wrapper-Funktion: nimmt eine Frage, ruft den Agent auf, gibt nur die finale Textantwort zurück

3. main.py geschrieben — die eigentliche FastAPI-App:

- CORSMiddleware eingebaut (erlaubt späteren Zugriff vom Frontend)
- QuestionRequest-Modell (Pydantic) — validiert eingehende Anfragen automatisch
- POST /ask — nimmt eine Frage, ruft ask_agent() auf, gibt Antwort zurück
- GET / — einfacher Health-Check-Endpunkt

4. requirements.txt geschrieben — Liste aller nötigen Pakete (fastapi, uvicorn, pydantic, python-dotenv, openai, chromadb, langchain, langchain-openai), damit auch ein fremder Server (später Render) weiß, was zu installieren ist
5. Lokal installiert: pip install -r backend/requirements.txt
6. Lokal gestartet: uvicorn main:app --reload (aus dem backend/-Ordner heraus)
7. Getestet über die automatische API-Doku (/docs) — POST /ask mit einer echten Frage ausprobiert, funktionierende, inhaltlich fundierte Antwort erhalten, CORS-Header bestätigt korrekt gesetzt

Frontend (React)

Frontend — Zusammenfassung von 0 bis fertig deployed

1. Projekt mit Vite aufgesetzt

   bash
   npm create vite@latest frontend -- --template react
   cd frontend
   npm install
   → Grundgerüst für ein React-Projekt (ohne TypeScript), Standard-Abhängigkeiten installiert

2. frontend/src/App.jsx geschrieben — die eigentliche Anwendung:

- Drei useState-Hooks: question (Texteingabe), answer (Backend-Antwort), loading (Ladezustand)
- handleAsk() — schickt bei Klick einen fetch-POST-Request an unser Backend, mit try/catch/finally für Fehlerbehandlung
- Einfaches UI: Textarea, Senden-Button (deaktiviert während des Ladens oder bei leerem Feld), Antwortanzeige 

3. Lokal getestet — npm run dev gestartet (Vite-Dev-Server, Port 5173/5174), im Browser Frage gestellt, komplette Verbindung Frontend → Backend → Agent → Antwort erfolgreich durchgespielt 
4. Git/GitHub eingerichtet (galt für das ganze Projekt, nicht nur Frontend):

- .gitignore erstellt (schließt venv/, node_modules/, .env, generierte Daten aus)
- Nach zwei fehlgeschlagenen Versuchen (.env und venv waren versehentlich in der Git-Historie, dadurch Timeout beim Push) sauber neu aufgesetzt: rm -rf .git → git init → add → commit
- Mit GitHub-Repository verbunden (git remote add origin ...) und gepusht 

5. Backend-URL im Code aktualisiert — von http://127.0.0.1:8000 (lokal) auf die echte Render-Backend-URL (https://health-fitness-qa-bot-backend.onrender.com) umgestellt, committet, gepusht 
6. Als "Static Site" auf Render deployed:

- Root Directory: frontend
- Build Command: npm install && npm run build (baut React zu statischen HTML/CSS/JS-Dateien)
- Publish Directory: dist
- Kein Start Command, keine Umgebungsvariable nötig (im Gegensatz zum Backend) 

7. Live getestet — echte Frage über die öffentliche Frontend-URL gestellt, Antwort kam korrekt vom deployten Backend zurück
